# TrustLens three-class evidence authenticity training

This notebook trains one EfficientNet-B0 to distinguish AUTHENTIC, TAMPERED, and SYNTHETIC images. It streams both public datasets, excludes source groups from both previous audit packs, writes outputs to Drive, and never replaces deployed models automatically.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib

drive.mount('/content/drive')

print('Select train_evidence_authenticity_streaming.py')
script_upload = files.upload()
assert 'train_evidence_authenticity_streaming.py' in script_upload

print('Select compact_provenance.csv')
compact_upload = files.upload()
assert 'compact_provenance.csv' in compact_upload

print('Select final_audit_provenance.csv')
audit_upload = files.upload()
assert 'final_audit_provenance.csv' in audit_upload

BACKEND_DIR = Path('/content/backend')
SCRIPT = BACKEND_DIR / 'scripts' / 'train_evidence_authenticity_streaming.py'
COMPACT_PROVENANCE = BACKEND_DIR / 'calibration' / 'compact_provenance.csv'
AUDIT_PROVENANCE = BACKEND_DIR / 'calibration' / 'final_audit_provenance.csv'
SCRIPT.parent.mkdir(parents=True, exist_ok=True)
COMPACT_PROVENANCE.parent.mkdir(parents=True, exist_ok=True)
SCRIPT.write_bytes(script_upload[SCRIPT.name])
COMPACT_PROVENANCE.write_bytes(compact_upload[COMPACT_PROVENANCE.name])
AUDIT_PROVENANCE.write_bytes(audit_upload[AUDIT_PROVENANCE.name])
digest = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
assert digest == '9f74763270413abf0f7f8299655ce2695b9897fe04bd99fbbbb1f4dba5fa3fc4', digest
print('Correct training script verified:', digest)

In [ ]:
%pip install -q "datasets>=4,<5" "scikit-learn>=1.5,<2" "tqdm>=4.66"

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess
import sys

OUTPUT_DIR = Path('/content/drive/MyDrive/TrustLens/trustlens_MJ-1/backend/training_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = OUTPUT_DIR / 'evidence_authenticity_forensic_multiview_candidate.pth'
REPORT = OUTPUT_DIR / 'evidence_authenticity_forensic_multiview_report.json'
command = [
    sys.executable, str(SCRIPT),
    '--output', str(CHECKPOINT),
    '--report', str(REPORT),
    '--exclude-provenance', str(COMPACT_PROVENANCE),
    '--exclude-provenance', str(AUDIT_PROVENANCE),
    '--train-per-class', '1600',
    '--validation-per-class', '300',
    '--epochs', '8',
    '--batch-size', '32',
    '--augmentation-profile', 'forensic',
]
subprocess.run(command, cwd=BACKEND_DIR, check=True)
print('Candidate:', CHECKPOINT)
print('Report:', REPORT)

In [ ]:
import json

report = json.loads(REPORT.read_text())
size_mb = CHECKPOINT.stat().st_size / 1024 / 1024
best = max(report['history'], key=lambda item: item['validation']['minimum_class_auroc'])
print('Checkpoint size:', round(size_mb, 2), 'MB')
print('Best minimum class AUROC:', round(report['best_validation_minimum_class_auroc'], 6))
print('Best per-class metrics:', json.dumps(best['validation']['per_class'], indent=2))
print('Excluded evaluation groups:', report['partition']['excluded_evaluation_groups'])
print('Promotion status:', report['promotion_status'])
assert size_mb > 10
assert report['partition']['excluded_evaluation_groups'] >= 180
assert report['promotion_status'] == 'CANDIDATE_REQUIRES_EXTERNAL_CALIBRATION'

## After training

Download `evidence_authenticity_three_class_candidate.pth` and `evidence_authenticity_three_class_report.json` from the Drive `training_outputs` directory. Do not deploy the checkpoint before a new independent TrustLens audit passes.